# 04 · Supplementary Figures 1–4

Reproduces all four supplementary figures from Rahnev (2025).

| Figure | Content |
|--------|---------|
| Supp Fig 1 | Validity and precision per measure (Haddara + Maniscalco) |
| Supp Fig 2 | Difficulty dependence for all 17 measures + d', criterion, confidence |
| Supp Fig 3 | Xue recoding before vs after (Haddara, Maniscalco, Shekhar) |
| Supp Fig 4 | Across-subject Pearson correlation matrices |

**Run notebook 02 first** to generate precomputed results.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import sys, os, warnings
warnings.filterwarnings('ignore')

REPO = os.path.abspath(os.path.join(os.getcwd(),
    '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
sys.path.insert(0, os.path.join(REPO, 'src'))
sys.path.insert(0, os.path.join(REPO, 'notebooks'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from analysis_core import (
    MEASURE_NAMES, N_MEASURES, ttest_1samp,
    preprocess_haddara, preprocess_maniscalco, preprocess_shekhar,
    preprocess_rouault, metas_altered_conf, xue_recode,
)
from metasignal.stdpy.compute_all import compute_all_measures

OUT   = os.path.join(REPO, 'notebooks', 'precomputed')
FIGS  = os.path.join(REPO, 'notebooks', 'figures')
os.makedirs(FIGS, exist_ok=True)

# Colour scheme matching good_colors_for_plotting.m
COLORS = {
    'haddara':    '#d55e00',
    'maniscalco': '#0072b2',
    'shekhar':    '#009e73',
    'rouault1':   '#e69f00',
    'rouault2':   '#cc79a7',
}

BIN_SIZES  = [50, 100, 200, 400]
PROP_ALTER = [0.02, 0.04, 0.06]

print('Environment ready.')

## Supplementary Figure 1 — Validity and Precision

**(a)** Validity and precision in the **Maniscalco** dataset (equivalent to paper Fig 1a which shows Haddara).  
Confidence for correct trials decreased by 1, confidence for incorrect trials increased by 1.  
Plot shows the drop in each measure in units of SD of that measure's fluctuations across bins.

**(b)** Average precision in SD units for each measure, averaged across 4 bin sizes and 3 corruption levels.

In [ ]:
def compute_precision_dataset(subjects, label):
    """Compute precision (SD-normalised drop under conf corruption) for a dataset.
    Returns prec_sd: (n_measures, n_bin_sizes, n_prop_altered) array.
    """
    prec_sd = np.full((N_MEASURES, len(BIN_SIZES), len(PROP_ALTER)), np.nan)

    for bi, bs in enumerate(BIN_SIZES):
        # collect raw and altered measures across all subjects × bins
        meas_raw = []
        meas_alt = [[] for _ in PROP_ALTER]
        for s in subjects:
            nr   = s['n_ratings']
            n    = len(s['stim'])
            n_bins = n // bs
            for b in range(n_bins):
                sl = slice(b*bs, (b+1)*bs)
                st, re, co = s['stim'][sl], s['resp'][sl], s['conf'][sl]
                raw = compute_all_measures(st, re, co, nr)
                meas_raw.append(raw)
                for ai, pa in enumerate(PROP_ALTER):
                    meas_alt[ai].append(metas_altered_conf(st, re, co, nr, pa))

        raw_arr = np.array(meas_raw)          # (total_bins, 20)
        sd_raw  = np.nanstd(raw_arr, axis=0)  # (20,)

        for ai in range(len(PROP_ALTER)):
            alt_arr = np.array(meas_alt[ai])
            drop    = raw_arr - alt_arr        # positive = measure went down
            drop_sd = np.nanmean(drop, axis=0) / (sd_raw + 1e-10)
            prec_sd[:, bi, ai] = drop_sd

        print(f'  {label} bin_size={bs}: done')

    return prec_sd


# Load or compute
ha_prec_path = os.path.join(OUT, 'haddara_precision.npz')
ma_prec_path = os.path.join(OUT, 'maniscalco_precision.npz')

if not os.path.exists(ha_prec_path):
    print('Computing Haddara precision (slow)...')
    ha = preprocess_haddara()
    ha_prec = compute_precision_dataset(ha, 'Haddara')
    np.savez(ha_prec_path, prec=ha_prec)
else:
    ha_prec = np.load(ha_prec_path)['prec']
    print('Loaded Haddara precision.')

if not os.path.exists(ma_prec_path):
    print('Computing Maniscalco precision (slow)...')
    ma = preprocess_maniscalco()
    ma_prec = compute_precision_dataset(ma, 'Maniscalco')
    np.savez(ma_prec_path, prec=ma_prec)
else:
    ma_prec = np.load(ma_prec_path)['prec']
    print('Loaded Maniscalco precision.')

print('Precision data ready. ha_prec shape:', ha_prec.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Supplementary Figure 1: Validity and Precision', fontsize=13, fontweight='bold')

# Panel (a): Maniscalco — precision for each bin size × corruption level
ax = axes[0]
ax.set_title('(a) Maniscalco dataset', fontweight='bold')
x = np.arange(1, N_MEASURES + 1)
for bi, bs in enumerate(BIN_SIZES):
    for ai, pa in enumerate(PROP_ALTER):
        vals = ma_prec[:, bi, ai]
        ax.plot(x, vals, alpha=0.6, linewidth=0.8,
                color=plt.cm.Blues(0.3 + 0.2*bi + 0.1*ai))

# Highlight average
avg = np.nanmean(ma_prec, axis=(1, 2))
ax.plot(x, avg, 'k-', linewidth=2, label='Mean')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(MEASURE_NAMES, rotation=90, fontsize=7)
ax.set_ylabel('Drop in measure (SD units)')
ax.set_xlabel('Measure')
ax.legend()

# Panel (b): Average precision both datasets
ax2 = axes[1]
ax2.set_title('(b) Average precision SD — Haddara vs Maniscalco', fontweight='bold')
for prec, label, color in [
    (ha_prec, 'Haddara (n=70)',    COLORS['haddara']),
    (ma_prec, 'Maniscalco (n=22)', COLORS['maniscalco']),
]:
    avg  = np.nanmean(prec, axis=(1, 2))         # mean over bin sizes and corruption levels
    sem  = np.nanstd(prec, axis=(1, 2)) / np.sqrt(len(BIN_SIZES) * len(PROP_ALTER))
    ax2.errorbar(x, avg, yerr=sem, fmt='o-', color=color, label=label,
                 capsize=3, linewidth=1.5, markersize=4)

ax2.axhline(np.nanmean(np.nanmean(ha_prec, axis=(1,2))[:16]),
            color=COLORS['haddara'], linestyle='--', alpha=0.5, label='_')
ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(MEASURE_NAMES, rotation=90, fontsize=7)
ax2.set_ylabel('Mean drop in measure (SD units)')
ax2.set_xlabel('Measure')
ax2.legend()

plt.tight_layout()
path = os.path.join(FIGS, 'supp_fig1_precision.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
print(f'Saved: {path}')
plt.show()

## Supplementary Figure 2 — Difficulty Dependence

Estimated metacognitive ability for all 17 measures + d', criterion, confidence at different difficulty levels in Shekhar, Rouault1, and Rouault2.

Rows: Traditional → Ratio → Diff → Model-based  
Columns: one measure each

In [ ]:
sh_diff_path  = os.path.join(OUT, 'shekhar_results.npz')
r1_diff_path  = os.path.join(OUT, 'rouault1_results.npz')
r2_diff_path  = os.path.join(OUT, 'rouault2_results.npz')

# Load or raise a clear error
if not all(os.path.exists(p) for p in [sh_diff_path, r1_diff_path, r2_diff_path]):
    print('Run 02_compute_measures.ipynb first to generate the precomputed files.')
else:
    sh_diff_full = np.load(sh_diff_path)['diff']  # (n_sub, 3, N_MEASURES)
    r1_diff = np.load(r1_diff_path)['diff']        # (n_sub, 2, N_MEASURES)
    r2_diff = np.load(r2_diff_path)['diff']

    n_rows, n_cols = 4, 5
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 14), squeeze=False)
    fig.suptitle('Supplementary Figure 2: Difficulty Dependence of Metacognitive Measures',
                 fontsize=12, fontweight='bold')

    datasets = [
        (sh_diff_full, [1, 2, 3], 'Shekhar', COLORS['shekhar']),    # 3 contrasts
        (r1_diff,      [0, 1],    'Rouault1', COLORS['rouault1']),   # low/high
        (r2_diff,      [0, 1],    'Rouault2', COLORS['rouault2']),
    ]

    for mi, mname in enumerate(MEASURE_NAMES[:20]):
        row, col = mi // n_cols, mi % n_cols
        ax = axes[row][col]
        ax.set_title(mname, fontsize=8, fontweight='bold')

        for diff_arr, x_vals, label, color in datasets:
            n_lvls = len(x_vals)
            means = [np.nanmean(diff_arr[:, li, mi]) for li in range(n_lvls)]
            sems  = [np.nanstd(diff_arr[:, li, mi]) / np.sqrt(np.sum(~np.isnan(diff_arr[:, li, mi])))
                     for li in range(n_lvls)]
            ax.errorbar(x_vals, means, yerr=sems, color=color,
                        marker='o', markersize=4, linewidth=1.5,
                        capsize=3, label=label)

        ax.set_xticks(x_vals if isinstance(x_vals[0], int) and x_vals[0] in [1,2,3] else [1,2])
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)

        # Significance annotation (paired t-test easy vs hard)
        if diff_arr.shape[1] == 3:
            delta = sh_diff_full[:, 2, mi] - sh_diff_full[:, 0, mi]
        else:
            delta = r1_diff[:, 1, mi] - r1_diff[:, 0, mi]
        t, df, p, *_ = ttest_1samp(delta)
        stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        ax.text(0.95, 0.95, stars, transform=ax.transAxes,
                ha='right', va='top', fontsize=8)

    # Legend in top-left panel
    axes[0][0].legend(fontsize=7, loc='lower right')

    plt.tight_layout()
    path = os.path.join(FIGS, 'supp_fig2_difficulty.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved: {path}')
    plt.show()

## Supplementary Figure 3 — Xue Recoding Before vs After

Shows metacognitive scores after recoding (Xue et al. method) alongside the raw scores before recoding for Haddara, Maniscalco, and Shekhar.

> Key finding: ~60% (Haddara), 55% (Maniscalco), 52% (Shekhar) of metacognitive scores were higher after recoding than before.

In [ ]:
def compute_raw_and_recode(subjects):
    """Returns (raw, recode1, recode2) each (n_sub, N_MEASURES)."""
    raw_all, r1_all, r2_all = [], [], []
    for s in subjects:
        nr = s['n_ratings']
        raw_all.append(compute_all_measures(s['stim'], s['resp'], s['conf'], nr))
        c1 = xue_recode(s['conf'], 1)
        c2 = xue_recode(s['conf'], 2)
        r1_all.append(compute_all_measures(s['stim'], s['resp'], c1, nr-1)
                      if not np.all(np.isnan(c1)) else np.full(N_MEASURES, np.nan))
        r2_all.append(compute_all_measures(s['stim'], s['resp'], c2, nr-1)
                      if not np.all(np.isnan(c2)) else np.full(N_MEASURES, np.nan))
    return np.array(raw_all), np.array(r1_all), np.array(r2_all)


# Load from precomputed or compute on demand
ha_res  = np.load(os.path.join(OUT, 'haddara_results.npz'))    if os.path.exists(os.path.join(OUT, 'haddara_results.npz'))    else None
ma_res  = np.load(os.path.join(OUT, 'maniscalco_results.npz')) if os.path.exists(os.path.join(OUT, 'maniscalco_results.npz')) else None
sh_res  = np.load(os.path.join(OUT, 'shekhar_results.npz'))    if os.path.exists(os.path.join(OUT, 'shekhar_results.npz'))    else None

if any(x is None for x in [ha_res, ma_res, sh_res]):
    print('Some precomputed files missing. Run 02_compute_measures.ipynb first.')
else:
    # bias arrays: (n_sub, 2, N_MEASURES); dim1 = [recode1, recode2]
    # average recode = mean of recode1 and recode2
    ha_avg_recode = np.nanmean(ha_res['bias'], axis=1)   # (n_sub, N_MEASURES)
    ma_avg_recode = np.nanmean(ma_res['bias'], axis=1)
    # Shekhar bias is per contrast: average first
    sh_bias_avg = np.nanmean(sh_res['bias'], axis=1)     # (n_sub, 2, N_MEASURES) after contract-avg
    sh_avg_recode = np.nanmean(sh_bias_avg, axis=1)      # (n_sub, N_MEASURES)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Supplementary Figure 3: Raw vs Recoded Metacognitive Scores',
                 fontsize=12, fontweight='bold')

    EXCLUDE_IDX = {MEASURE_NAMES.index("d'"), MEASURE_NAMES.index('Criterion')}
    x = [i for i in range(N_MEASURES) if i not in EXCLUDE_IDX]
    x_labels = [MEASURE_NAMES[i] for i in x]

    for ax, (raw_path, recode, n, label, color) in zip(axes, [
        (ha_res['raw'], ha_avg_recode, 70, 'Haddara',    COLORS['haddara']),
        (ma_res['raw'], ma_avg_recode, 22, 'Maniscalco', COLORS['maniscalco']),
        (sh_res['diff'][:,1,:] if 'diff' in sh_res else None, sh_avg_recode, 20, 'Shekhar', COLORS['shekhar']),
    ]):
        ax.set_title(f'{label} (n={n})', fontweight='bold')
        if raw_path is None: continue

        raw_mean = np.nanmean(raw_path[:, x], axis=0)
        raw_sem  = np.nanstd(raw_path[:, x], axis=0) / np.sqrt(n)
        rec_mean = np.nanmean(recode[:, x], axis=0)
        rec_sem  = np.nanstd(recode[:, x], axis=0) / np.sqrt(n)

        xi = np.arange(len(x))
        ax.errorbar(xi, raw_mean, yerr=raw_sem, fmt='k-', linewidth=2.5,
                    capsize=2, label='Raw (before recode)', zorder=3)
        ax.errorbar(xi, rec_mean, yerr=rec_sem, fmt='o-', color=color,
                    linewidth=1.5, capsize=2, label='After Xue recode', alpha=0.8)

        # Mark significant t-tests
        for xi_i, mi in enumerate(x):
            delta = recode[:, mi] - raw_path[:, mi]
            t, df, p, *_ = ttest_1samp(delta)
            if not np.isnan(p) and p < 0.05:
                ax.axvline(xi_i, color='gray', linewidth=0.3, alpha=0.4)

        ax.set_xticks(xi)
        ax.set_xticklabels(x_labels, rotation=90, fontsize=7)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
        ax.set_ylabel('Mean measure ± SEM')
        ax.legend(fontsize=8)

    plt.tight_layout()
    path = os.path.join(FIGS, 'supp_fig3_xue_recode.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved: {path}')
    plt.show()

## Supplementary Figure 4 — Across-Subject Correlations

Pearson correlation matrix across all 20 measures, computed separately for Haddara, Maniscalco, and Shekhar.  
Star symbols indicate p < 0.05 (uncorrected).

In [ ]:
if any(x is None for x in [ha_res, ma_res, sh_res]):
    print('Run 02_compute_measures.ipynb first.')
else:
    # Use the 'raw' measures for each dataset
    # For Shekhar, average raw measures over contrasts
    sh_raw = np.nanmean(sh_res['diff'], axis=1)  # (n_sub, 20)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('Supplementary Figure 4: Across-Subject Correlations Between Measures',
                 fontsize=12, fontweight='bold')

    for ax, (raw, label, n) in zip(axes, [
        (ha_res['raw'], 'Haddara (n=70)',     70),
        (ma_res['raw'], 'Maniscalco (n=22)',  22),
        (sh_raw,        'Shekhar (n=20)',      20),
    ]):
        # Pearson correlation matrix
        R = np.full((N_MEASURES, N_MEASURES), np.nan)
        P = np.full((N_MEASURES, N_MEASURES), np.nan)
        for i in range(N_MEASURES):
            for j in range(N_MEASURES):
                xi, xj = raw[:, i], raw[:, j]
                valid = ~np.isnan(xi) & ~np.isnan(xj)
                if valid.sum() >= 3:
                    r, p = stats.pearsonr(xi[valid], xj[valid])
                    R[i, j], P[i, j] = r, p

        im = ax.imshow(R, vmin=-1, vmax=1, cmap='RdYlBu_r', aspect='auto')
        ax.set_title(label, fontweight='bold')
        ax.set_xticks(range(N_MEASURES))
        ax.set_yticks(range(N_MEASURES))
        ax.set_xticklabels(MEASURE_NAMES, rotation=90, fontsize=6)
        ax.set_yticklabels(MEASURE_NAMES, fontsize=6)

        # Star for significant correlations (p < 0.05)
        for i in range(N_MEASURES):
            for j in range(N_MEASURES):
                if not np.isnan(P[i, j]) and P[i, j] < 0.05 and i != j:
                    ax.text(j, i, '★', ha='center', va='center',
                            fontsize=5, color='white' if abs(R[i,j]) > 0.6 else 'black')

        plt.colorbar(im, ax=ax, shrink=0.7, label='Pearson r')

    plt.tight_layout()
    path = os.path.join(FIGS, 'supp_fig4_correlations.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved: {path}')
    plt.show()

In [ ]:
print('\nFigures saved to:', FIGS)
for f in sorted(os.listdir(FIGS)):
    print(f'  {f}')